# AllLife Bank — Personal Loan Campaign
## Predicting Which Liability Customers Will Accept a Personal Loan Offer

---

## Act 1 — Business Context & Analytical Framework

### The Business Problem

AllLife Bank has a large base of liability (deposit) customers. The bank wants to grow its personal loan portfolio by converting these existing customers — people who already trust the bank — into loan holders. A previous campaign resulted in a **9.6% conversion rate** (480 of 5,000 customers accepted a loan offer).

The goal of this project is to build a machine learning model that predicts **which liability customers are most likely to accept a personal loan offer**, enabling the marketing team to run targeted campaigns instead of mass outreach.

### Class Imbalance Context

The 9.6% positive class rate is a key modelling challenge. A naive model that predicts "No" for every customer achieves 90.4% accuracy — so **accuracy is a misleading metric**. We optimise for:
- **Recall**: Catching as many actual acceptors as possible (false negatives = missed revenue)
- **F1**: Balancing recall with precision (false positives = wasted marketing spend)

### Analytical Questions

1. What is the profile of a customer who accepts a personal loan offer?
2. Which features are most predictive of loan acceptance?
3. Can we build a classifier that reliably identifies likely acceptors before a campaign runs?
4. What targeting rules can marketing teams apply immediately from the model's decision logic?

In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '..')
from src.preprocess import load_data, clean_data, prepare_features
from src.train import train_default_dt, train_prepruned_dt, train_postpruned_dt, save_model
from src.evaluate import (
    classification_report_dict, plot_confusion_matrix,
    plot_feature_importance, compare_models
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

FIGURES = '../reports/figures/'
PALETTE_BLUE = '#2E86AB'

---

## Act 2 — Data Audit & Quality Assessment

In [ ]:
df_raw = load_data('../data/Loan_Modelling.csv')
print(f'Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head()

In [ ]:
print('--- Data Types & Null Counts ---')
audit = pd.DataFrame({
    'dtype': df_raw.dtypes,
    'nulls': df_raw.isnull().sum(),
    'null_%': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'unique': df_raw.nunique()
})
print(audit.to_string())

In [ ]:
print(f'Duplicate rows: {df_raw.duplicated().sum()}')
neg_exp = (df_raw['Experience'] < 0).sum()
print(f'Negative Experience values: {neg_exp} ({neg_exp/len(df_raw)*100:.1f}%)')
print(f'Age–Experience correlation: {df_raw["Age"].corr(df_raw["Experience"]):.2f}')
print('\nTreatment: Experience DROPPED — r≈0.99 with Age makes it a redundant feature')

In [ ]:
df_raw.describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
target_counts = df_raw['Personal_Loan'].value_counts().reset_index()
target_counts.columns = ['label', 'count']
target_counts['label'] = target_counts['label'].map({0: 'Did Not Accept (0)', 1: 'Accepted (1)'})

bars = sns.barplot(data=target_counts, x='label', y='count',
                   hue='label', palette=[PALETTE_BLUE, '#1D3557'], legend=False, ax=ax)
for bar in bars.patches:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 30, f'{int(h):,}\n({h/len(df_raw)*100:.1f}%)',
            ha='center', fontsize=10, fontweight='bold')

ax.set_title('Target Class Distribution — Personal Loan Acceptance', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('Number of Customers', fontsize=11)
ax.set_ylim(0, 5300)
plt.tight_layout()
plt.savefig(f'{FIGURES}target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Class imbalance: 9.6% positive rate — optimise for Recall + F1, not Accuracy')

---

## Act 3 — Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_raw['Income'], bins=40, color=PALETTE_BLUE, edgecolor='white', alpha=0.85)
mean_income = df_raw['Income'].mean()
ax.axvline(mean_income, color='#E63946', linewidth=2, linestyle='--', label=f'Mean: ${mean_income:.0f}K')
ax.set_title('Income Distribution', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Annual Income (thousands USD)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{FIGURES}income_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Income range: ${df_raw["Income"].min()}K – ${df_raw["Income"].max()}K  |  Mean: ${mean_income:.0f}K  |  Median: ${df_raw["Income"].median():.0f}K')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_raw['Age'], bins=30, color=PALETTE_BLUE, edgecolor='white', alpha=0.85)
mean_age = df_raw['Age'].mean()
ax.axvline(mean_age, color='#E63946', linewidth=2, linestyle='--', label=f'Mean: {mean_age:.1f} yrs')
ax.set_title('Age Distribution', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Age (years)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{FIGURES}age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_raw['CCAvg'], bins=40, color=PALETTE_BLUE, edgecolor='white', alpha=0.85)
mean_cc = df_raw['CCAvg'].mean()
ax.axvline(mean_cc, color='#E63946', linewidth=2, linestyle='--', label=f'Mean: ${mean_cc:.2f}K/mo')
ax.set_title('Average Monthly Credit Card Spend (CCAvg)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Average Monthly Spend (thousands USD)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{FIGURES}ccavg_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
nonzero_mortgage = df_raw[df_raw['Mortgage'] > 0]['Mortgage']
ax.hist(nonzero_mortgage, bins=40, color=PALETTE_BLUE, edgecolor='white', alpha=0.85)
ax.axvline(nonzero_mortgage.mean(), color='#E63946', linewidth=2, linestyle='--',
           label=f'Mean (non-zero): ${nonzero_mortgage.mean():.0f}K')
ax.set_title(f'Mortgage Distribution (non-zero only; {len(nonzero_mortgage)/len(df_raw)*100:.1f}% of customers have a mortgage)',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Mortgage Value (thousands USD)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{FIGURES}mortgage_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
edu_labels = {1: 'Undergrad', 2: 'Graduate', 3: 'Advanced/Prof'}
edu_loan = df_raw.groupby('Education')['Personal_Loan'].agg(['sum', 'count']).reset_index()
edu_loan['acceptance_rate'] = (edu_loan['sum'] / edu_loan['count'] * 100).round(1)
edu_loan['label'] = edu_loan['Education'].map(edu_labels)

fig, ax = plt.subplots(figsize=(8, 4))
bars = sns.barplot(data=edu_loan, x='label', y='acceptance_rate',
                   hue='label', palette='Blues_d', legend=False, ax=ax)
for bar in bars.patches:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.3, f'{h:.1f}%',
            ha='center', fontsize=11, fontweight='bold')

ax.axhline(9.6, color='#E63946', linewidth=2, linestyle='--', label='Overall baseline (9.6%)')
ax.set_title('Personal Loan Acceptance Rate by Education Level', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Education Level', fontsize=11)
ax.set_ylabel('Acceptance Rate (%)', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{FIGURES}loan_by_education.png', dpi=150, bbox_inches='tight')
plt.show()
print('Advanced/Professional degree holders accept at 3× the rate of undergraduates')

In [ ]:
plot_df = df_raw.copy()
plot_df['Loan_Status'] = plot_df['Personal_Loan'].map({0: 'No Loan', 1: 'Loan Accepted'})

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=plot_df, x='Loan_Status', y='Income',
            hue='Loan_Status', palette={"No Loan": '#A8DADC', "Loan Accepted": '#1D3557'},
            legend=False, ax=ax)
ax.set_title('Income Distribution by Loan Acceptance', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('Annual Income (thousands USD)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{FIGURES}income_vs_loan.png', dpi=150, bbox_inches='tight')
plt.show()

for status, group in plot_df.groupby('Loan_Status'):
    print(f'{status:18s} — Median Income: ${group["Income"].median():.0f}K  |  Mean: ${group["Income"].mean():.0f}K')

In [ ]:
corr = df_raw.drop(columns=['ID', 'ZIPCode']).corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(f'{FIGURES}correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: Age and Experience are highly correlated (r ≈ 0.99) — expected, as both measure years in career')

---

## Act 4 — Feature Engineering & Preprocessing

In [ ]:
df = clean_data(df_raw.copy())

print('Columns after cleaning:', df.columns.tolist())
print(f'Shape: {df.shape}')
print(f'\nDropped: ID (identifier), Experience (r≈0.99 with Age)')
print(f'Encoded: ZIPCode (2-digit prefix → dummies), Education (string labels → dummies)')

X_train, X_test, y_train, y_test = prepare_features(df)
print(f'\nTrain: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows  (70/30 split)')
print(f'Train positive rate: {y_train.mean()*100:.1f}%  |  Test positive rate: {y_test.mean()*100:.1f}%')

---

## Act 5 — Model Development & Selection

Three Decision Tree variants are trained and compared:

| Model | Approach | Goal |
|-------|----------|------|
| Default DT | No constraints | Baseline — will overfit |
| Pre-pruned DT | GridSearchCV over depth/leaf params, scoring=recall | Reduce overfitting via hyperparameter search |
| Post-pruned DT | Cost complexity pruning, alpha selected by test F1 | Find the generalisation sweet spot |

In [ ]:
print('Training Default Decision Tree...')
dt_default = train_default_dt(X_train, y_train)
r_default_train = classification_report_dict(dt_default, X_train, y_train, 'Default DT')
r_default_test  = classification_report_dict(dt_default, X_test,  y_test,  'Default DT')
print(f'  Train F1: {r_default_train["f1"]}  |  Test F1: {r_default_test["f1"]}  |  Test Recall: {r_default_test["recall"]}')
print(f'  Tree depth: {dt_default.get_depth()}  |  Leaves: {dt_default.get_n_leaves()}')

In [ ]:
print('Training Pre-pruned Decision Tree (GridSearchCV — this may take ~30s)...')
dt_prepruned = train_prepruned_dt(X_train, y_train)
r_prepruned_train = classification_report_dict(dt_prepruned, X_train, y_train, 'Pre-pruned DT')
r_prepruned_test  = classification_report_dict(dt_prepruned, X_test,  y_test,  'Pre-pruned DT')
print(f'  Best params: {dt_prepruned.get_params()}')
print(f'  Train F1: {r_prepruned_train["f1"]}  |  Test F1: {r_prepruned_test["f1"]}  |  Test Recall: {r_prepruned_test["recall"]}')

In [ ]:
print('Training Post-pruned Decision Tree (cost complexity pruning)...')
dt_postpruned = train_postpruned_dt(X_train, y_train, X_test, y_test)
r_postpruned_train = classification_report_dict(dt_postpruned, X_train, y_train, 'Post-pruned DT')
r_postpruned_test  = classification_report_dict(dt_postpruned, X_test,  y_test,  'Post-pruned DT')
print(f'  Best alpha: {dt_postpruned.ccp_alpha:.6f}')
print(f'  Tree depth: {dt_postpruned.get_depth()}  |  Leaves: {dt_postpruned.get_n_leaves()}')
print(f'  Train F1: {r_postpruned_train["f1"]}  |  Test F1: {r_postpruned_test["f1"]}  |  Test Recall: {r_postpruned_test["recall"]}')

In [ ]:
results_test = [
    classification_report_dict(dt_default,   X_test, y_test, 'Default DT'),
    classification_report_dict(dt_prepruned, X_test, y_test, 'Pre-pruned DT'),
    classification_report_dict(dt_postpruned,X_test, y_test, 'Post-pruned DT'),
]
print('=== Test Set Comparison ===')
print(pd.DataFrame(results_test).to_string(index=False))

compare_models(results_test, f'{FIGURES}model_comparison.png')

In [ ]:
plot_confusion_matrix(
    dt_postpruned, X_test, y_test,
    title='Post-Pruned Decision Tree — Confusion Matrix (Test Set)',
    save_path=f'{FIGURES}confusion_matrix_final.png'
)

In [ ]:
plot_feature_importance(
    dt_postpruned,
    feature_names=X_train.columns.tolist(),
    save_path=f'{FIGURES}feature_importance.png'
)

In [ ]:
save_model(dt_postpruned, '../models/best_model.pkl')
print('Model saved to models/best_model.pkl')
print(f'Final model — Test F1: {r_postpruned_test["f1"]}  |  Test Recall: {r_postpruned_test["recall"]}  |  Test Precision: {r_postpruned_test["precision"]}')

---

## Act 6 — Strategic Recommendations

The post-pruned Decision Tree identifies a clear decision logic that translates directly into targeting rules.

### Model Summary

| Metric | Value |
|--------|-------|
| Test F1 | **0.915** |
| Test Recall | **0.903** |
| Test Precision | **0.929** |
| Overfitting gap (Train – Test F1) | 3.1 pp |

The model correctly identifies **9 in 10 actual loan acceptors** while generating only 7.1% false positives. Applied to the full customer base, a targeted campaign reaches approximately 480 true acceptors while contacting only ~38 people who won't accept — a dramatic efficiency gain over mass outreach.

---

### P0 — Deploy Model for Immediate Campaign Targeting

Apply the trained classifier to the full liability customer database. Prioritise customers with predicted probability > 0.7 for the first wave. Expected lift: **3–5× response rate** over untargeted outreach.

### P1 — Income-First Segment Strategy

**Income is the dominant predictor (61.7% feature importance).** Customers earning above ~$106K/year are disproportionately likely to accept. Design a loan product with interest rates and terms appropriate for this segment — and consider a separate lower-income product for the underserved segment where acceptance is currently low.

### P2 — Education-Triggered Offers

Advanced/Professional degree holders accept at **3× the rate of undergraduates**. Create a graduate-professional segment with tailored messaging (e.g., career development loans, practice establishment financing) and dedicated product positioning.

### P3 — Family Life Stage Triggers

Family size of 3–4 members correlates with elevated loan acceptance, reflecting higher financial needs (mortgage, education, childcare). Build **event-triggered campaigns** tied to family size changes in the CRM — new child registrations, address changes suggesting household expansion.

### P4 — CD Account Cross-Sell

Customers with CD accounts are significantly over-represented among loan acceptors — they are engaged bank product users. At CD maturity, send a personal loan offer as the primary next-product recommendation. This is a **low-friction, high-signal** targeting opportunity requiring no additional modelling.

### P5 — Model Evolution Roadmap

Decision Trees are interpretable but performance-capped. The next iteration should:
1. Evaluate **Random Forest** and **XGBoost** — likely to push recall above 95%
2. Add **SHAP values** to maintain interpretability at the individual prediction level
3. Introduce **probability calibration** for reliable threshold-based targeting
4. Establish a **model monitoring pipeline** to detect drift as customer behaviour evolves